# AIST 2026 — Turkish NLU segmentation (Kaggle runner)

**Before running:** Notebook → Settings → **Accelerator: GPU T4 x1**, **Internet: On**.

The full 3×3×3 matrix fine-tunes base encoders on ~12k MASSIVE-tr examples in
minutes per cell and fits one Kaggle session.

In [ ]:
# 1) Get the code. Replace with your repo, or add the project as a Kaggle Dataset.
!git clone https://github.com/<you>/turkish-nlu-segmentation.git
%cd turkish-nlu-segmentation
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) Sanity + data + mandatory alignment guard (must be ALL PASS)
!python tests/test_core.py
!python scripts/download_data.py
!python scripts/validate_alignment.py --data data/raw/tr-TR.jsonl --models berturk mbert xlmr

In [ ]:
# 3) A quick single cell first to confirm GPU timing, then the full matrix.
!python -m src.train --config configs/base.yaml --set model_name=dbmdz/bert-base-turkish-cased segmentation=morphological seed=42
# Full sweep (27 runs):
!python scripts/run_matrix.py

In [ ]:
# 4) Tables + significance + morphology-aware error analysis
!python scripts/aggregate.py
!python scripts/error_report.py --predictions outputs/berturk_morphological_seed42/test_predictions.json --data data/raw/tr-TR.jsonl
!python scripts/aggregate.py --compare outputs/xlmr_morphological_seed42/test_predictions.json outputs/xlmr_native_seed42/test_predictions.json

In [ ]:
# 5) Save outputs before the session ends (Kaggle wipes /kaggle/working otherwise)
!zip -r outputs.zip outputs && echo 'Download outputs.zip from the Output tab'